#  Step 3: Hybrid RAG (No BitsAndBytes)

This notebook implements a Hybrid RAG system using **Gemma 2 (2B)** in native bfloat16.
Use `pipeline.py build_rag` for the canonical end-to-end workflow.
It intelligently switches between Pandas (Stats) and Vector Search (Stories).

In [ ]:
# Dependencies
%pip install -qU pandas faiss-cpu sentence-transformers transformers

In [ ]:
import os
import glob

# --- DEBUG CELL ---
print(f" Current Working Directory: {os.getcwd()}")
if os.path.exists("data"):
    files = glob.glob("data/*.json")
    print(f" found 'data' folder with {len(files)} JSON files.")
else:
    print(" 'data' folder NOT found in current directory.")
    
# Check absolute path backup
abs_path = "/mnt/c/Users/pragn/Downloads/Gemma_finetune/data"
if os.path.exists(abs_path):
    files = glob.glob(f"{abs_path}/*.json")
    print(f" found absolute path '{abs_path}' with {len(files)} JSON files.")

📂 Current Working Directory: /home/pragn
✅ found 'data' folder with 0 JSON files.
✅ found absolute path '/mnt/c/Users/pragn/Downloads/Gemma_finetune/data' with 1169 JSON files.


In [ ]:
import pandas as pd
import json
import glob
import torch
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# --- CONFIG ---
MATCH_DATA_DIR = "data"
MODEL_ID = "google/gemma-2-2b-it"

# Robust Path Logic
if not os.path.exists(MATCH_DATA_DIR) or not glob.glob(f"{MATCH_DATA_DIR}/*.json"):
    # Try absolute path
    abs_path = "/mnt/c/Users/pragn/Downloads/Gemma_finetune/data"
    if os.path.exists(abs_path):
        MATCH_DATA_DIR = abs_path
        print(f" Switched to absolute path: {MATCH_DATA_DIR}")
    else:
        # Try relative to the notebook loaction if different
        # Sometimes CWD is not where the notebook is
        print(f" WARNING: Data directory '{MATCH_DATA_DIR}' not found!")

print(f" Initializing Hybrid RAG with {MODEL_ID}...")

🔄 Switched to absolute path: /mnt/c/Users/pragn/Downloads/Gemma_finetune/data
🚀 Initializing Hybrid RAG with google/gemma-2-2b-it...


In [ ]:
# 1. Load Structured Data (The "SQL" part)
rows = []
files = glob.glob(f"{MATCH_DATA_DIR}/*.json")
print(f" Scanning {MATCH_DATA_DIR}... Found {len(files)} files.")

if len(files) == 0:
    # Last ditch attempt: walk properly
    # Maybe it is C:/Users... (Windows style) instead of /mnt/c/...
    win_path = "C:/Users/pragn/Downloads/Gemma_finetune/data"
    if os.path.exists(win_path):
        print(f"🔄 Trying Windows path: {win_path}")
        files = glob.glob(f"{win_path}/*.json")
        # Update MATCH_DATA_DIR
        MATCH_DATA_DIR = win_path

for f in files:
    try:
        with open(f, 'r') as file:
            data = json.load(file)
            info = data['info']
            
            row = {
                "date": info['dates'][0] if 'dates' in info else "Unknown",
                "team1": info['teams'][0],
                "team2": info['teams'][1],
                "winner": info.get('outcome', {}).get('winner', 'No Result'),
                "venue": info.get('venue', "Unknown"),
                "player_of_match": info.get('player_of_match', ["None"])[0],
                "file_path": f
            }
            rows.append(row)
    except Exception as e:
        # print(f" Error reading {f}: {e}")
        pass

if not rows:
    raise ValueError(f" No data loaded from {MATCH_DATA_DIR}! Check the Debug Cell above.")

df = pd.DataFrame(rows)
print(f" Structured DB Loaded: {len(df)} matches")
df.head()

📂 Scanning /mnt/c/Users/pragn/Downloads/Gemma_finetune/data... Found 1169 files.
📊 Structured DB Loaded: 1169 matches


,date,team1,team2,winner,venue,player_of_match,file_path
0,2017-04-05,Sunrisers Hyderabad,Royal Challengers Bangalore,Sunrisers Hyderabad,"Rajiv Gandhi International Stadium, Uppal",Yuvraj Singh,/mnt/c/Users/pragn/Downloads/Gemma_finetune/da...
1,2017-04-06,Rising Pune Supergiant,Mumbai Indians,Rising Pune Supergiant,Maharashtra Cricket Association Stadium,SPD Smith,/mnt/c/Users/pragn/Downloads/Gemma_finetune/da...
2,2017-04-07,Gujarat Lions,Kolkata Knight Riders,Kolkata Knight Riders,Saurashtra Cricket Association Stadium,CA Lynn,/mnt/c/Users/pragn/Downloads/Gemma_finetune/da...
3,2017-04-08,Kings XI Punjab,Rising Pune Supergiant,Kings XI Punjab,Holkar Cricket Stadium,GJ Maxwell,/mnt/c/Users/pragn/Downloads/Gemma_finetune/da...
4,2017-04-08,Royal Challengers Bangalore,Delhi Daredevils,Royal Challengers Bangalore,M.Chinnaswamy Stadium,KM Jadhav,/mnt/c/Users/pragn/Downloads/Gemma_finetune/da...


In [ ]:
# 2. Load Vector Index (The "Semantic" part)
embedder = SentenceTransformer("all-MiniLM-L6-v2")
match_summaries = [f"{r['team1']} vs {r['team2']} at {r['venue']}, {r['winner']} won." for r in rows]
embeddings = embedder.encode(match_summaries, convert_to_numpy=True)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print(" Vector Index Built")

📖 Vector Index Built


In [19]:
# 3. Load LLM (bfloat16 - No bnb)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=200)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


In [ ]:
def router(query):
    """
    Decides if the query needs 'STATS' (Pandas) or 'STORY' (Vector).
    """
    prompt_text = f"""
    Classify the following cricket query into exactly one category: 'STATS' or 'STORY'.
    
    Query: "Who won the match between MI and CSK in 2019?"
    Category: STATS
    
    Query: "Describe the excitement of the last over."
    Category: STORY
    
    Query: "{query}"
    Category:
    """
    
    messages = [{"role": "user", "content": prompt_text}]
    out = pipe(tokenizer.apply_chat_template(messages, tokenize=False), max_new_tokens=5)
    # Robust router parsing
    response = out[0]['generated_text']
    if "Category:" in response:
        decision = response.split("Category:")[-1].strip().upper()
    else:
        decision = response.split("model")[-1].strip().upper()
        
    return "STATS" if "STATS" in decision else "STORY"

def query_pandas(query):
    # Simple keyword search
    hits = df[df.apply(lambda row: row.astype(str).str.contains(query, case=False).any(), axis=1)]
    if not hits.empty:
        return hits.to_string(index=False)
    return "No exact stats found."

def query_vector(query):
    emb = embedder.encode([query])
    D, I = index.search(emb, k=3)
    return match_summaries[I[0][0]] 

def hybrid_rag(user_query):
    category = router(user_query)
    print(f" Routing to: {category}")
    
    context = ""
    if category == "STATS":
        context = query_pandas(user_query)
    else:
        context = query_vector(user_query)
        
    # UPDATED: Encouraging full sentence answer
    final_prompt = f"""
    You are a helpful cricket assistant. Answer the user's question in a complete, natural sentence using the context provided.
    
    Question: {user_query}
    Context: {context}
    
    Answer (in a full sentence):
    """
    messages = [{"role": "user", "content": final_prompt}]
    out = pipe(tokenizer.apply_chat_template(messages, tokenize=False), max_new_tokens=100)
    
    # Robust extraction: look for the last model turn or 'Answer:'
    # Gemma specific delimiter
    return out[0]['generated_text'].split("<start_of_turn>model")[-1].strip()

In [21]:
# TEST IT
q1 = "Who won the match between Sunrisers and KKR?"
print(f"\nQ: {q1}")
print(f"A: {hybrid_rag(q1)}")

q2 = "Describe a match with high tension"
print(f"\nQ: {q2}")
print(f"A: {hybrid_rag(q2)}")


Q: Who won the match between Sunrisers and KKR?
🔀 Routing to: STORY
A: <bos><start_of_turn>user
You are a helpful cricket assistant. Answer the user's question in a complete, natural sentence using the context provided.

    Question: Who won the match between Sunrisers and KKR?
    Context: Sunrisers Hyderabad vs Kolkata Knight Riders at Feroz Shah Kotla, Sunrisers Hyderabad won.

    Answer (in a full sentence):<end_of_turn>
Sunrisers Hyderabad won the match against Kolkata Knight Riders at Feroz Shah Kotla.

Q: Describe a match with high tension
🔀 Routing to: STORY
A: <bos><start_of_turn>user
You are a helpful cricket assistant. Answer the user's question in a complete, natural sentence using the context provided.

    Question: Describe a match with high tension
    Context: Delhi Daredevils vs Pune Warriors at Feroz Shah Kotla, Pune Warriors won.

    Answer (in a full sentence):<end_of_turn>
*The match between Delhi Daredevils and Pune Warriors was a nail-biter, with the tension